# wym のモデルを作成する


In [ ]:
dataset_root_dir = "../../data/lemon/datasets"
out_root_dir = "../../data/wym/model"
dataset_names = [
    "structured_amazon_google",
    "structured_beer",
    "structured_dblp_acm",
    "structured_dblp_google_scholar",
    "structured_fodors_zagat",
    "structured_walmart_amazon",
    "structured_itunes_amazon",
    "dirty_dblp_acm",
    "dirty_dblp_google_scholar",
    "dirty_walmart_amazon",
    "dirty_itunes_amazon",
    "textual_abt_buy",
    "textual_company",
]

In [ ]:
TARGET_DATASET_ID = 11
GPU_ID = 1

In [ ]:
BATCH_SIZE = 512

In [ ]:
# torchモジュールの読み込み前に、利用できるGPUを指定しておく
## これをやらないと、システム内のＧＰＵすべてを利用してしまう
import os

os.environ["CUDA_VISIBLE_DEVICES"] = f"{GPU_ID}"

import torch

print("CUDA =", torch.cuda.is_available())
print("CUDA DEVICES =", torch.cuda.device_count())
print("CUDA CURRENT DEVICE_ID = ", torch.cuda.current_device())

In [ ]:
# transformers の tokenizer を並列実行で呼び出すか（dead lockしてしまう）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# Set Random Seeds and Reproducibility
import random

import numpy as np


def set_seed(seed: int):
    """
    Helper function for reproducible behavior to set the seed in ``random``, ``numpy``, ``torch``
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

## データセットを読み込む

In [ ]:
from pine.dataset import load_dataset
from lemon.utils.datasets import SplittedDataset


def test_load_dataset():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    display(dataset.test.records.a.head())
    display(dataset.test.records.b.head())
    display(dataset.test.record_id_pairs.head())
    display(dataset.test.labels.head())
    print(dataset.test.records.a.dtypes)
    print("Test DATA SIZE =", len(dataset.test.record_id_pairs))


test_load_dataset()

## モデル作成

### データ変換

In [ ]:
import pandas as pd
import unicodedata


def remove_accents(input_str: str) -> str:
    """
    Unicode のアクセント記号を削除して基本的な a-z に変換
    Args:
        input_str (str): 入力文字列
    Returns:
        str: アクセント記号を削除した文字列
    """
    # Unicode 正規化 (NFKD) によって分解
    normalized_str = unicodedata.normalize('NFKD', input_str)
    # 分解されたアクセント記号を除去
    return ''.join(c for c in normalized_str if not unicodedata.combining(c))


def normalize_str(df: pd.DataFrame) -> pd.DataFrame:
    """
    文字列の正規化を行う。ウムラウト系を削除。大文字小文字を統一する。
    Args:
        df (pd.DataFrame): 文字列を含むDataFrame
    Returns:
        pd.DataFrame: 正規化されたDataFrame
    """
    # 置換ルールを辞書で定義
    # replace_dict = {
    #     "â": "a",
    #     "ã": "a",
    #     "á": "a",
    #     "å": "a",
    #     "ä": "a",
    #     "é": "e",
    #     "ï": "i",
    #     "ñ": "n",
    #     "ö": "o",
    #     "û": "u",
    #     "ù": "u",
    #     "ü": "u",
    #     "ÿ": "y",
    #     "ß": "ss",
    #     "ç": "c",
    #     "ô": "o",
    #     "ó": "o",
    #     "õ": "o",
    #     "ë": "e",
    #     "ì": "i",
    #     "í": "i",
    #     "ò": "o",
    # }
    # replace_dict.update({k.upper():v.upper() for k,v in replace_dict.items()})

    # 置換関数を定義
    def replace_chars(value):
        if isinstance(value, str):  # 文字列のみ処理
            value = remove_accents(value).lower()
        return value

    # DataFrame に適用
    df = df.applymap(replace_chars)
    return df


def convert_dataset_to_wym(
    records_left: pd.DataFrame,
    records_right: pd.DataFrame,
    record_id_pairs: pd.DataFrame,
    labels: pd.Series = None,
) -> pd.DataFrame:
    """
    lemonのデータセットをWYMの入力形式に変換する
    WWMの入力形式は以下のDatarrameを返す。
    id,left_id,right_id,label,left_*(左側のカラム), right_右側のカラム

    Args:
        records_left (pd.DataFrame): 左側のレコード
        records_right (pd.DataFrame): 右側のレコード
        record_id_pairs (pd.DataFrame): レコードIDのペア
        labels (pd.DataFrame, optional): ラベル. Defaults to None.
    Returns:
        pd.DataFrame: WYMの入力形式
    """
    # record_leftとrecord_rightの内容のうち文字列を変更する
    records_left = normalize_str(records_left)
    records_right = normalize_str(records_right)

    # カラム名を変更
    df = record_id_pairs.rename(columns={"a.rid": "left_id", "b.rid": "right_id"})
    # ラベルがあれば結合
    if labels is not None:
        df = pd.merge(df, labels.astype(int), left_index=True, right_index=True)

    # レコードIDのペアを結合
    df = pd.merge(
        df,
        records_left.add_prefix("left_"),
        left_on="left_id",
        right_index=True,
    )
    df = pd.merge(
        df, records_right.add_prefix("right_"), left_on="right_id", right_index=True
    )

    # indexをidという名前のカラムに変換する
    df = df.reset_index().rename(columns={"pid": "id"})

    return df


def test_convert_dataset_to_wym():
    dataset_name = dataset_names[TARGET_DATASET_ID]
    dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
    df = convert_dataset_to_wym(
        dataset.train.records.a,
        dataset.train.records.b,
        dataset.train.record_id_pairs,
        dataset.train.labels,
    )
    display(df.head())
    print("DATA SIZE =", len(df))


test_convert_dataset_to_wym()


### 学習

In [ ]:
from wym.wym import Wym

In [ ]:
from warnings import simplefilter

simplefilter(action="ignore", category=FutureWarning)

In [ ]:
import pathlib


def train_wym(dataset: SplittedDataset, batch_size: int = 512, out_dir: str = "."):
    # データセットの変換
    train_df = convert_dataset_to_wym(
        dataset.train.records.a,
        dataset.train.records.b,
        dataset.train.record_id_pairs,
        dataset.train.labels,
    )
    val_df = convert_dataset_to_wym(
        dataset.val.records.a,
        dataset.val.records.b,
        dataset.val.record_id_pairs,
        dataset.val.labels,
    )

    exclude_attrs = ["id", "left_id", "right_id", "label"]

    # モデルの初期化
    wym = Wym(
        df=train_df,
        exclude_attrs=exclude_attrs,
        batch_size=batch_size,
        reset_networks=True,
        model_files_path=out_dir,
    )

    # モデルの学習
    wym.fit(
        train_df[wym.columns_to_use],
        train_df["label"],
        val_df[wym.columns_to_use],
        val_df["label"],
    )

    return


In [ ]:
dataset_name = dataset_names[TARGET_DATASET_ID]
dataset: SplittedDataset = load_dataset(dataset_name, dataset_root_dir)
out_dir_path = pathlib.Path(out_root_dir) / dataset_name
out_dir_path.mkdir(parents=True, exist_ok=True)
train_wym(dataset, BATCH_SIZE, str(out_dir_path))